In [1]:
import subprocess
import re
import pandas as pd
import shutil
from tqdm import tqdm
import openpyxl
import os
import gc

In [2]:
# tree = "propose_tree"
tree = "original_tree"

In [3]:
# quartus_bin = r"C:\altera_standard\25.1std\quartus\bin64"

# # Compile
# subprocess.run(
#     [f"{quartus_bin}\\quartus_sh.exe", "--flow", "compile", "generic_propose_tree"],
#         cwd=rf"C:\Users\Rafa\Desktop\HCR\Generics_Tree\{tree}\v32",
#     check=True
# )

# # Timing
# subprocess.run(
#     [f"{quartus_bin}\\quartus_sta.exe", "-t", "script.tcl"],
#     cwd=rf"C:\Users\Rafa\Desktop\HCR\Generics_Tree\{tree}\v32",
#     check=True
# )

## Diretorios

In [4]:
quartus_bin = r"C:\altera_standard\25.1std\quartus\bin64"
base_dir = rf"C:\Users\Rafa\Desktop\HCR\Generics_Tree\{tree}"

# m_values = [16, 32, 64, 128, 256, 512]
m_values = [256, 512]
n_values = [16, 24, 32, 48, 64, 96, 128, 256,  512] 

# m_values = [256]
# n_values = [256] 

## Functions

In [5]:
# =========================
# altera n no VHDL
# =========================
def set_n(vhdl_file, n):
    with open(vhdl_file, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    text = re.sub(
        r"n\s*:\s*integer\s*:=\s*\d+",
        f"n : integer := {n}",
        text
    )

    with open(vhdl_file, "w", encoding="utf-8") as f:
        f.write(text)


# =========================
# acha nome do projeto .qpf
# =========================
def find_project_name(project_dir):
    qpf_files = [f for f in os.listdir(project_dir) if f.endswith(".qpf")]

    if not qpf_files:
        raise FileNotFoundError(f"Nenhum arquivo .qpf encontrado em {project_dir}")

    return os.path.splitext(qpf_files[0])[0]


# =========================
# roda quartus
# =========================
def run_quartus(project_dir):
    shutil.rmtree(os.path.join(project_dir, "db"), ignore_errors=True)
    shutil.rmtree(os.path.join(project_dir, "incremental_db"), ignore_errors=True)

    project_name = find_project_name(project_dir)

    try:
        subprocess.run(
            [
                os.path.join(quartus_bin, "quartus_sh.exe"),
                "--flow",
                "compile",
                project_name
            ],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True
        )

        subprocess.run(
            [
                os.path.join(quartus_bin, "quartus_sta.exe"),
                "-t",
                "script.tcl"
            ],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            f"Quartus failed in {project_dir} with return code {e.returncode}\n"
            f"stdout:\n{e.stdout}\n"
            f"stderr:\n{e.stderr}"
        ) from e


# =========================
# parse dos paths
# =========================
def parse_paths(file):
    paths = []
    current = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Path #\d+: Delay is\s+([\d\.]+)", line)
            if m:
                if current:
                    paths.append(current)

                current = {
                    "delay": float(m.group(1)),
                    "ic": None,
                    "cell": None
                }
                continue

            if current is None:
                continue

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["ic"] = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["cell"] = float(m.group(1))

    if current:
        paths.append(current)

    df = pd.DataFrame(paths)

    if df.empty:
        return df

    return df.dropna()


# =========================
# calcula métricas
# =========================
def compute_metrics(df):
    critical = df.loc[df["delay"].idxmax()]

    return {
        "Delay_CP": critical["delay"],
        "IC_CP": critical["ic"],
        "CELL_CP": critical["cell"],

        "Delay_MAX": df["delay"].max(),
        "IC_MAX": df["ic"].max(),
        "CELL_MAX": df["cell"].max(),

        "Delay_MEAN": df["delay"].mean(),
        "IC_MEAN": df["ic"].mean(),
        "CELL_MEAN": df["cell"].mean()
    }


# =========================
# extrai caminho crítico
# =========================
def extract_critical_path(file):
    delay = None
    ic = None
    cell = None
    from_node = None
    to_node = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Delay is\s+([\d\.]+)", line)
            if m:
                delay = float(m.group(1))

            m = re.search(r";\s*From Node\s*;\s*(.*?)\s*;", line)
            if m:
                from_node = m.group(1)

            m = re.search(r";\s*To Node\s*;\s*(.*?)\s*;", line)
            if m:
                to_node = m.group(1)

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                ic = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                cell = float(m.group(1))

    return {
        "delay": delay,
        "ic": ic,
        "cell": cell,
        "from": from_node,
        "to": to_node
    }


In [6]:
# =========================
# loop principal
# =========================
resultados = []

for num_vec in m_values:

    project_dir = os.path.join(base_dir, f"v{num_vec}")
    vhdl_file = os.path.join(project_dir, "vhdl", f"sum{num_vec}xn.vhd")
    report_file = os.path.join(project_dir, "io_paths.txt")
    worst_file = os.path.join(project_dir, "worst_path.txt")

    resultados_m = []

    print(f"\n==============================")
    print(f" Rodando projeto com {num_vec} vetores")
    print(f"==============================")

    for n in tqdm(n_values):

        df = None
        metrics = None
        cp = None

        try:
            print(f"\nRodando num_vec = {num_vec}, n = {n}")

            set_n(vhdl_file, n)
            run_quartus(project_dir)

            df = parse_paths(report_file)

            if df.empty:
                raise ValueError("Nenhum path encontrado em io_paths.txt")

            metrics = compute_metrics(df)
            cp = extract_critical_path(worst_file)

            linha = {
                "num_vec": num_vec,
                "n": n,

                "Delay_CP": cp["delay"],
                "IC_CP": cp["ic"],
                "CELL_CP": cp["cell"],

                "From_Node": cp["from"],
                "To_Node": cp["to"],

                "Delay_MAX": metrics["Delay_MAX"],
                "IC_MAX": metrics["IC_MAX"],
                "CELL_MAX": metrics["CELL_MAX"],

                "Delay_MEAN": metrics["Delay_MEAN"],
                "IC_MEAN": metrics["IC_MEAN"],
                "CELL_MEAN": metrics["CELL_MEAN"],
            }

            resultados.append(linha)
            resultados_m.append(linha)

        except Exception as e:
            print(f"Erro em num_vec={num_vec}, n={n}: {e}")
            continue

        finally:
            for name in ("df", "metrics", "cp"):
                globals().pop(name, None)
            gc.collect()

    # salva um arquivo separado para cada número de vetores
    df_m = pd.DataFrame(resultados_m).sort_values("n")

    csv_m = os.path.join(base_dir, f"timing_v{num_vec}.csv")
    xlsx_m = os.path.join(base_dir, f"timing_v{num_vec}.xlsx")

    df_m.to_csv(csv_m, index=False)
    df_m.to_excel(xlsx_m, index=False)

    print(f"\nArquivos salvos para {num_vec} vetores:")
    print(csv_m)
    print(xlsx_m)


 Rodando projeto com 256 vetores


  0%|          | 0/9 [00:00<?, ?it/s]


Rodando num_vec = 256, n = 16


 11%|█         | 1/9 [02:19<18:33, 139.14s/it]


Rodando num_vec = 256, n = 24


...


Rodando num_vec = 512, n = 256


 89%|████████▉ | 8/9 [3:23:24<57:00, 3420.36s/it]


Rodando num_vec = 512, n = 512


100%|██████████| 9/9 [10:24:30<00:00, 4163.36s/it] 

Erro em num_vec=512, n=512: Quartus failed in C:\Users\Rafa\Desktop\HCR\Generics_Tree\original_tree\v512 with return code 3
stdout:
Info: *******************************************************************
Info: Running Quartus Prime Shell
    Info: Version 25.1std.0 Build 1129 10/21/2025 SC Standard Edition
    Info: Copyright (C) 2025  Altera Corporation. All rights reserved.
    Info: Your use of Altera Corporation's design tools, logic functions 
    Info: and other software and tools, and any partner logic 
    Info: functions, and any output files from any of the foregoing 
    Info: (including device programming or simulation files), and any 
    Info: associated documentation or information are expressly subject 
    Info: to the terms and conditions of the Altera Program License 
    Info: Subscription Agreement, the Altera Quartus Prime License Agreement,
    Info: the Altera IP License Agreement, or other applicable license
    Info: agreement, including, without limitation,

In [7]:
# =========================

# salva resultado geral
# =========================
final_df = pd.DataFrame(resultados).sort_values(["num_vec", "n"])

csv_final = os.path.join(base_dir, "timing_all_vectors.csv")
xlsx_final = os.path.join(base_dir, "timing_all_vectors.xlsx")

final_df.to_csv(csv_final, index=False)
final_df.to_excel(xlsx_final, index=False)

print("\nResultado final:")
print(final_df)

print("\nArquivos gerais salvos:")
print(csv_final)
print(xlsx_final)


Resultado final:
    num_vec    n  Delay_CP   IC_CP  CELL_CP From_Node   To_Node  Delay_MAX  \
0       256   16    12.657   9.467    3.190    a81[1]   sum[15]     12.657   
1       256   24    11.839   8.745    3.094  a104[17]   sum[22]     11.839   
2       256   32    13.008   9.160    3.848   a111[3]   sum[31]     13.008   
3       256   48    15.228  11.800    3.428  a118[33]   sum[47]     15.228   
4       256   64    20.773  16.042    4.731    a98[6]   sum[63]     20.773   
5       256   96    22.491  18.606    3.885   a38[67]   sum[95]     22.491   
6       256  128    33.547  28.086    5.461   a17[58]  sum[118]     33.547   
7       256  256    43.874  35.898    7.976    a5[36]  sum[253]     43.874   
8       512   16    13.705  11.010    2.695   a137[8]   sum[15]     13.705   
9       512   24    15.944  12.016    3.928    a19[7]   sum[22]     15.944   
10      512   32    19.005  15.035    3.970     a3[3]   sum[31]     19.005   
11      512   48    20.808  16.845    3.963   